# SSW60 Downstream Evaluation

We evaluate 4 out of our 5 iNatSounds-trained models on the SSW60 audio test set 
(because we lost the ConvNeXt 100% run weights and were not able to re-run the training). 
Again, we follow the same protocol as [Chasmai et al. (2024)](https://arxiv.org/pdf/2506.00343):

- No fine-tuning: our models are not fine-tuned on SSW60, we directly use them with 
their iNatSounds fine-tuning we conducted
- Class mask: As per Chasmai et al., logits for the 5,509 species absent from SSW60 are 
set to minus infinity effectively masking species that are not present in the dataset
- Sliding-window: the same sliding window methodology that we used for the iNatSounds 
dataset is applied here
- Reported Metrics: we simply show Top-1 and Top-5 accuracy at the sample-level (exactly 
as Chasmai et al.)

Note: all parameter counts in this notebook are shown with the full 5,569 classification 
head for all models (hence the much higher parameter count vs. backbone only parameters).


In [ ]:
# Paths to the necessary directories (available in the google drive link)
SSW60_SPEC_DIR   = "ssw60_test_spectrograms" # flat directory of .npy spectrogram files
SSW60_CSV        = "ssw60/audio_ml.csv" # SSW60 metadata (contains split, label columns)
SSW60_TAXA_CSV   = "ssw60/taxa.csv" # SSW60 taxa (enables us to translate labels -> 
# to the inat_taxon_id)
INAT_TRAIN_JSON  = "train.json" # iNatSounds train.json (for taxon -> class idx)

# Model Configurations
# Note: we must import the weights of each model in this notebooks directory to be able 
# to run this pipeline
MODELS = [
    {
        "name":     "MobileNetV3",
        "builder":  "mobilenet",
        "weights":  "mobilenetv3_50epoch_weights.pt",
        "img_size": 224,
    },
    {
        "name":     "EfficientNet-B0",
        "builder":  "efficientnet_b0",
        "weights":  "efficientnet-b0_50epochs_weights.pt",
        "img_size": 224,
    },
    {
        "name":     "ConvNeXt",
        "builder":  "convnext_base",
        "weights":  "convnext_epoch49_25pct_run.pt",
        "img_size": 224,
    },
    {
        "name":     "Swin-V2-B",
        "builder":  "swin_v2_b",
        "weights":  "swin_v2_b_epoch050_weights.pt",
        "img_size": 256,
    },
]

NUM_CLASSES   = 5569 # iNatSounds training classes
TARGET_LENGTH = 512 # c.3 seconds at hop=128 / sr=22050
TEST_STRIDE   = 256 # c.1.5 seconds
BATCH_SIZE    = 8 # recordings per batch (each yields several windows)
SUB_BATCH     = 64 # sub-batch for forward pass (again, ensure VRAM control)
NUM_WORKERS   = 0 

SPEC_MEAN = 0.6569
SPEC_STD  = 0.1786


In [ ]:
# Install key dependencies

!pip install timm scikit-learn tqdm pandas -q

In [ ]:
# Check Cuda (ensures that we are indeed using the GPU)

import torch
print(torch.cuda.is_available())  # should be True
print(torch.version.cuda)         # should show 12.1

True
12.1


In [ ]:
# Import dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import os
import timm
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

# Again, check the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti
VRAM: 4.3 GB


## SSW60 Class Mask

We map the 60 SSW60 species to their indices in the 5569-class iNatSounds model
via `inat_taxon_id` which is the name of the species (this is shared key between 
`taxa.csv` from SSW60 and `train.json` from iNatSounds).

In [ ]:
# Print the name from iNatSounds to check
with open("train.json") as f:
    data = json.load(f)

# Print first category to see all available fields
print(data["categories"][0])

{'id': 0, 'name': 'Hodotermes mossambicus', 'common_name': 'Northern Harvester Termite', 'supercategory': 'Insecta', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Blattodea', 'family': 'Hodotermitidae', 'genus': 'Hodotermes', 'specific_epithet': 'mossambicus', 'audio_dir_name': '00000_Animalia_Arthropoda_Insecta_Blattodea_Hodotermitidae_Hodotermes_mossambicus', 'evaluation': 0}


In [ ]:
# Print the scientific name from SSW60 to check
taxa = pd.read_csv("ssw60/taxa.csv")
print(taxa.columns.tolist())
print(taxa.head(3))

['label', 'species_code', 'inat_taxon_id', 'inat2021_label', 'nabirds_labels', 'common_name', 'scientific_name', 'family', 'order']
   label species_code  inat_taxon_id  inat2021_label nabirds_labels  \
0      0       cangoo           7089            3226         57 457   
1      1       wooduc           7107            3188     81 314 613   
2      2      mallar3           6930            3201    102 317 616   

    common_name     scientific_name                                  family  \
0  Canada Goose   Branta canadensis  Anatidae (Ducks, Geese, and Waterfowl)   
1     Wood Duck          Aix sponsa  Anatidae (Ducks, Geese, and Waterfowl)   
2       Mallard  Anas platyrhynchos  Anatidae (Ducks, Geese, and Waterfowl)   

          order  
0  Anseriformes  
1  Anseriformes  
2  Anseriformes  


In [ ]:
# We build the mask for iNatSounds trained models using the name data shown above
def build_ssw60_mask(taxa_csv, train_json, num_classes=5569):
    # Read and record the names from SSW60
    taxa = pd.read_csv(taxa_csv)
    ssw60_label_to_sciname = dict(zip(taxa["label"], taxa["scientific_name"]))

    # get the names in iNatSounds
    with open(train_json) as f:
        data = json.load(f)
    sciname_to_inat_idx = {c["name"]: c["id"] for c in data["categories"]}

    # Create the mask
    mask = torch.zeros(num_classes, dtype=torch.bool)
    ssw60_label_to_inat_idx = {}
    missing = []

    # Populate the mask only marking species present in both SSW60 and iNatSounds
    for ssw60_label, sciname in ssw60_label_to_sciname.items():
        if sciname in sciname_to_inat_idx:
            inat_idx = sciname_to_inat_idx[sciname]
            mask[inat_idx] = True
            ssw60_label_to_inat_idx[int(ssw60_label)] = inat_idx
        else:
            missing.append(sciname)

    # Check if the species count matches
    print(f"SSW60 species mapped: {mask.sum().item()}/60")
    if missing:
        print(f"Warning: {len(missing)} species were not found in train.json: {missing}")

    return mask, ssw60_label_to_inat_idx

# Call the build mask function
ssw60_mask, ssw60_label_to_inat_idx = build_ssw60_mask(
    SSW60_TAXA_CSV, INAT_TRAIN_JSON, num_classes=NUM_CLASSES
)

SSW60 species mapped: 60/60


## SSW60 Dataset

We build a class which reads the .npy spectrograms (one per recording) and returns all 
sliding the windows per file, this pipeline is the same as iNatSounds in evaluation mode.

In [ ]:
class SSW60Dataset(Dataset):
    """SSW60 audio test set. Returns all sliding windows per recording.
    Labels are iNatSounds class indices so they are directly comparable to our model 
    outputs.
    """
    
    def __init__(self, spec_dir, csv_path, ssw60_label_to_inat_idx,
                 target_length=512, test_stride=256, img_size=224, split="test"):
        self.spec_dir    = spec_dir
        self.target_length = target_length
        self.test_stride   = test_stride
        self.img_size      = img_size

        df = pd.read_csv(csv_path)
        df = df[df["split"] == split].reset_index(drop=True)
        
        self.entries = []
        skipped = 0
        for _, row in df.iterrows():
            asset_id   = int(row["asset_id"])
            ssw60_label = int(row["label"])
            if ssw60_label not in ssw60_label_to_inat_idx:
                skipped += 1
                continue
            npy_path = os.path.join(spec_dir, f"{asset_id}.npy")
            if not os.path.exists(npy_path):
                skipped += 1
                continue
            self.entries.append({
                "path":  npy_path,
                "label": ssw60_label_to_inat_idx[ssw60_label],
                "name":  str(asset_id),
            })
        print(f"SSW60 {split}: {len(self.entries)} recordings loaded, {skipped} skipped")

    def __len__(self):
        return len(self.entries)

    def _dense_prediction(self, spec):
        """All sliding windows of target_length with stride test_stride.
        Input: (128, T) uint8. Output: (W, 128, target_length) uint8.
        Again, same as iNatSounds.
        """
        time_len = spec.shape[1]
        if time_len <= self.target_length:
            pad = self.target_length - time_len
            spec = np.pad(spec, ((0, 0), (pad // 2, pad - pad // 2)), mode="constant")
            time_len = spec.shape[1]
        num = (time_len - self.target_length + self.test_stride - 1) // self.test_stride
        pad = self.test_stride * num - time_len + self.target_length
        spec = np.pad(spec, ((0, 0), (pad // 2, pad - pad // 2)), mode="constant")
        windows = []
        for i in range(num + 1):
            start = i * self.test_stride
            if start + self.target_length > spec.shape[1]:
                break
            windows.append(spec[:, start:start + self.target_length])
        return np.stack(windows)

    def _to_tensor(self, windows, img_size):
        """(W, 128, T) uint8 -> (W, 3, img_size, img_size) normalised."""
        x = torch.from_numpy(windows.astype(np.float32) / 255.0)
        x = x.unsqueeze(1).repeat(1, 3, 1, 1)               # (W, 3, 128, T)
        x = F.interpolate(x, size=(img_size, img_size),
                          mode="bilinear", align_corners=False)
        return (x - SPEC_MEAN) / SPEC_STD

    def __getitem__(self, idx):
        e    = self.entries[idx]
        spec = np.load(e["path"])                            # (128, T) uint8
        windows = self._dense_prediction(spec)               # (W, 128, target_length)
        imgs    = self._to_tensor(windows, self.img_size)    # (W, 3, img_size, img_size)
        labels  = torch.tensor([e["label"]] * imgs.shape[0], dtype=torch.long)
        return imgs, labels, e["name"]

# We define a function to stack all the outputed sliding windows
def ssw60_collate(batch):
    """Stack variable-length window batches; track which windows belong to which recording."""
    imgs   = torch.cat([b[0] for b in batch], dim=0)
    labels = torch.cat([b[1] for b in batch], dim=0)
    names  = []
    for b in batch:
        names.extend([b[2]] * b[0].shape[0])
    return imgs, labels, names


## Evaluation Function

Here we use the Chasmai et al's protocol i.e. we average softmax across windows per file, 
then argmax over the 60 unmasked species. We report sample-level Top-1 and Top-5
(so that our reported figures are consistent with the paper's SSW60 reporting).

In [ ]:
@torch.no_grad()
def evaluate_ssw60(model, loader, mask, sub_batch_size=64):
    """Run SSW60 evaluation.

    Returns dict with sample-level top1, top5 (matching paper's SSW60 metric).
    """
    model.eval()
    softmax   = nn.Softmax(dim=-1)
    mask_dev  = mask.to(device)

    # Buffers: name -> (sum of softmax probs, count)
    buf_sum   = {}
    buf_count = {}
    buf_label = {}

    for specs, labels, names in tqdm(loader, desc="Eval", leave=False):
        specs  = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        all_logits = []
        for s in range(0, specs.size(0), sub_batch_size):
            with torch.amp.autocast(device.type):
                out = model(specs[s:s + sub_batch_size])
            all_logits.append(out.float())
        logits = torch.cat(all_logits, dim=0)

        # Mask out the 5509 species not in SSW60
        logits = logits.masked_fill(~mask_dev.unsqueeze(0), float("-inf"))
        probs  = softmax(logits).cpu()
        labels = labels.cpu()

        for i, name in enumerate(names):
            if name not in buf_sum:
                buf_sum[name]   = probs[i].clone()
                buf_count[name] = 1
                buf_label[name] = labels[i].item()
            else:
                buf_sum[name]  += probs[i]
                buf_count[name] += 1

    # Compute sample-level Top-1 and Top-5
    correct_top1 = 0
    correct_top5 = 0
    total = len(buf_sum)

    for name in buf_sum:
        avg_probs = buf_sum[name] / buf_count[name]
        gt        = buf_label[name]
        correct_top1 += int(avg_probs.argmax().item() == gt)
        correct_top5 += int(gt in torch.topk(avg_probs, 5).indices.tolist())

    return {
        "top1":  100.0 * correct_top1 / total,
        "top5":  100.0 * correct_top5 / total,
        "n_recordings": total,
    }


## Building the models

This mirrors the `get_model()` function exactly so the architectures we build
here match the ones used during training.

- `mobilenet`: uses `torchvision.models.mobilenet_v3_large` with head replaced at `classifier[3]`
- `efficientnet_b0`: uses `torchvision.models.efficientnet_b0` with head replaced at `classifier[1]`
- `convnext_base`: uses `timm.create_model("convnext_base.fb_in22k_ft_in1k", ...)`
- `swin_v2_b`: uses `timm.create_model("swinv2_base_window12to16_192to256.ms_in22k_ft_in1k", ...)`


In [ ]:
import torchvision
import timm


def build_model(builder, num_classes):
    """Construct our models with iNatSounds-trained backbone, head replaced for `num_classes`.
    This mirrors `get_model()` so that our saved state_dicts load cleanly.
    We note that no pretrained weights are loaded here.
    """
    if builder == "mobilenet":
        # torchvision MobileNetV3-Large: classifier = [Linear(960,1280), Hardswish,
        # Dropout, Linear(1280, 1000)]. Index 3 is the final classifier.
        model = torchvision.models.mobilenet_v3_large(weights=None)
        model.classifier[3] = nn.Linear(1280, num_classes)

    elif builder == "efficientnet_b0":
        # torchvision EfficientNet-B0: classifier = [Dropout(0.2), Linear(1280, 1000)].
        # Index 1 is the final classifier.
        model = torchvision.models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(1280, num_classes)

    elif builder == "convnext_base":
        model = timm.create_model(
            "convnext_base.fb_in22k_ft_in1k",
            pretrained=False, num_classes=num_classes,
        )

    elif builder == "swin_v2_b":
        model = timm.create_model(
            "swinv2_base_window12to16_192to256.ms_in22k_ft_in1k",
            pretrained=False, num_classes=num_classes,
        )

    else:
        raise ValueError(f"Unknown builder: {builder}")
    
    return model


## Testing Weight Loading

Before running eval, we make sure there are no issues with weight loading. This allows 
us to confirm that each active model's `timm_name` actually matches its saved
checkpoint.

In [ ]:
import os
import torch

for cfg in MODELS:
    print(f"\n[{cfg['name']}]  weights={cfg['weights']}  builder={cfg['builder']}")
    if not os.path.exists(cfg['weights']):
        print(f"  Fail: weight file not found at '{cfg['weights']}' (cwd={os.getcwd()})")
        continue

    # Inspect the checkpoint
    state = torch.load(cfg['weights'], map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    n_params_ckpt = sum(v.numel() for v in state.values() if torch.is_tensor(v)) / 1e6
    sample_keys   = list(state.keys())[:3] + ['...'] + list(state.keys())[-2:]
    print(f"  checkpoint: {len(state)} tensors, {n_params_ckpt:.1f}M params")
    print(f"  first/last keys: {sample_keys}")

    # Build the model and try to load
    try:
        model = build_model(cfg['builder'], NUM_CLASSES)
    except Exception as e:
        print(f"  Fail: build_model failed: {type(e).__name__}: {e}")
        continue
    n_params_model = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  built model: {n_params_model:.1f}M params")

    missing, unexpected = model.load_state_dict(state, strict=False)
    if not missing and not unexpected:
        print("  Success: strict load worked, builder matches the checkpoint")
    else:
        print(f"  Fail: {len(missing)} missing keys, {len(unexpected)} unexpected keys")
        if missing[:3]:
            print(f"    missing (first 3): {missing[:3]}")
        if unexpected[:3]:
            print(f"    unexpected (first 3): {unexpected[:3]}")
        print("    Fail: architecture mismatch; check how this model was trained")

    del model, state



[MobileNetV3]  weights=mobilenetv3_50epoch_weights.pt  builder=mobilenet
  checkpoint: 312 tensors, 11.4M params
  first/last keys: ['features.0.0.weight', 'features.0.1.weight', 'features.0.1.bias', '...', 'classifier.3.weight', 'classifier.3.bias']
  built model: 11.3M params
  Success: strict load worked, builder matches the checkpoint

[EfficientNet-B0]  weights=efficientnet-b0_50epochs_weights.pt  builder=efficientnet_b0
  checkpoint: 360 tensors, 11.2M params
  first/last keys: ['features.0.0.weight', 'features.0.1.weight', 'features.0.1.bias', '...', 'classifier.1.weight', 'classifier.1.bias']
  built model: 11.1M params
  Success: strict load worked, builder matches the checkpoint

[ConvNeXt]  weights=convnext_epoch49_25pct_run.pt  builder=convnext_base
  checkpoint: 344 tensors, 93.3M params
  first/last keys: ['stem.0.weight', 'stem.0.bias', 'stem.1.weight', '...', 'head.fc.weight', 'head.fc.bias']
  built model: 93.3M params
  Success: strict load worked, builder matches th

## Run the Test Set Evaluation

We run all four models on the SSW60 test set.

In [ ]:
import gc
import time

all_results = {}

for cfg in MODELS:
    print(f"\n{'='*55}")
    print(f"Model: {cfg['name']} | img_size={cfg['img_size']}")
    print(f"{'='*55}")
    
    # Dataset and Loader
    dataset = SSW60Dataset(
        spec_dir              = SSW60_SPEC_DIR,
        csv_path              = SSW60_CSV,
        ssw60_label_to_inat_idx = ssw60_label_to_inat_idx,
        target_length         = TARGET_LENGTH,
        test_stride           = TEST_STRIDE,
        img_size              = cfg["img_size"],
    )
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        collate_fn=ssw60_collate,
    )
    
    # We now load the models
    model = build_model(cfg["builder"], NUM_CLASSES)
    state = torch.load(cfg["weights"], map_location=device, weights_only=False)
    model.load_state_dict(state)
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Loaded {cfg['weights']} | {n_params:.1f}M params")
    
    # We evaluate the models and keep track of time
    t0 = time.time()
    results = evaluate_ssw60(model, loader, ssw60_mask, sub_batch_size=SUB_BATCH)
    elapsed = time.time() - t0
    
    print(f"Top-1: {results['top1']:.1f}%  |  Top-5: {results['top5']:.1f}%  "
          f"| {results['n_recordings']} recordings | {elapsed:.0f}s")
    
    all_results[cfg["name"]] = {**results, "builder": cfg["builder"],
                                 "img_size": cfg["img_size"], "weights": cfg["weights"]}
    
    # We make sure to free the VRAM before the next model to avoid our run crashing
    del model, dataset, loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# We print a summary of all our model results
print("\n" + "="*55)
print("SUMMARY")
print("="*55)
print(f"{'Model':<20} {'Top-1':>8} {'Top-5':>8}")
print("-"*40)
for name, r in all_results.items():
    print(f"{name:<20} {r['top1']:>7.1f}% {r['top5']:>7.1f}%")


Model: MobileNetV3 | img_size=224
SSW60 test: 1264 recordings loaded, 0 skipped
Loaded mobilenetv3_50epoch_weights.pt | 11.3M params


Top-1: 73.2%  |  Top-5: 87.1%  | 1264 recordings | 25s

Model: EfficientNet-B0 | img_size=224
SSW60 test: 1264 recordings loaded, 0 skipped
Loaded efficientnet-b0_50epochs_weights.pt | 11.1M params


Top-1: 74.7%  |  Top-5: 89.2%  | 1264 recordings | 34s

Model: ConvNeXt | img_size=224
SSW60 test: 1264 recordings loaded, 0 skipped
Loaded convnext_epoch49_25pct_run.pt | 93.3M params


Top-1: 70.3%  |  Top-5: 85.8%  | 1264 recordings | 811s

Model: Swin-V2-B | img_size=256
SSW60 test: 1264 recordings loaded, 0 skipped
Loaded swin_v2_b_epoch050_weights.pt | 92.6M params


Top-1: 79.8%  |  Top-5: 92.2%  | 1264 recordings | 1816s

SUMMARY
Model                   Top-1    Top-5
----------------------------------------
MobileNetV3             73.2%    87.1%
EfficientNet-B0         74.7%    89.2%
ConvNeXt                70.3%    85.8%
Swin-V2-B               79.8%    92.2%


In [ ]:
import json as jl

with open("ssw60_results.json", "w") as f:
    jl.dump(all_results, f, indent=2)
print("Saved to ssw60_results.json")

Saved to ssw60_results.json
